# Reasoning Prompts: When Thinking Helps, and What It Costs

**A loan-eligibility checker tested across four prompting strategies: Plain · Step-by-Step (CoT) · Step-Back · Self-Consistency (Voting)**

Session 14 focused on *what* you say to a model. This session explores *whether asking a model to think actually helps*. The same loan-eligibility problem is tested using four different reasoning strategies:

1. **Plain** — ask for only the final answer.
2. **Step-by-Step (CoT)** — ask the model to reason through the problem before answering.
3. **Step-Back** — identify the relevant rules or formulas first, then solve the case.
4. **Self-Consistency (Voting)** — solve the problem several times and use the majority answer.

The experiments use ten loan-eligibility cases with independently calculated ground-truth answers. For each strategy, we measure:

- **Accuracy** — how often the model reaches the correct answer
- **Tokens** — how much model output is used
- **Time** — how long the request takes
- **Cost** — the estimated cost at larger scale

The goal is to understand the **accuracy–cost–latency trade-off** rather than simply choosing the strategy with the highest accuracy.

### Environment & Budget Notes

- Run the notebook from the project repository using `uv`.
- The model and API configuration are loaded from `.env`.
- Model responses are cached so that unchanged experiments can be re-run without making the same API request again.
- Token usage and timing are recorded by the notebook's experiment meter.
- Results can vary by model and provider, so the tables produced by this notebook should be used as the final measurements.

## 1. Setup

In [4]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path

try:
    from providers_utils import groq_client, ollama_client
    from prompt_utils import PromptSpec, init_work_repo, save_to_library
    from prompt_tasks import TASK1, TASK2, TASK3, classify_spec
    from reasoning_tasks import (
        DEMO, LOAN_CASES, LOAN_POLICY, OTHER, STRETCH,
        classify_cot_spec, problem_card, score_classifier_lab
    )
    from reasoning_utils import (
    DiskCache, Lab, Meter,
    accuracy_markdown, case_table,
    check_s15_work, comparison_table,
    pass_rate, pick_hardest,
    pick_local_chat_model, pick_trouble,
    run_strategy, save_accuracy_table,
    strategy_summary, verdict, with_scratchpad,
    solve_plain, solve_cot,
    solve_stepback, solve_self_consistency
)
except ImportError as e:
    raise ImportError(
        f"{e}. This notebook uses files from Sessions 11, 14 and 15: "
        "run 'git pull' in the course folder, then restart the kernel."
    ) from e

MODEL = os.getenv("MODEL")

cache, meter = DiskCache(Path(r"D:\Projects\GenAI-Agents-Work\.s15_cache.json")), Meter()
client = None

if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()

lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None
ready = lab is not None

print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)

if not ready:
    print(
        "No key in .env: the sections that call the model will say 'skipped'. "
        "Everything else still runs."
    )

results = {}

hardest = LOAN_CASES[4]

def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw)
    results[(problem.id, a.strategy)] = a
    return a

def rows(
    problems=LOAN_CASES,
    strategies=("plain", "cot", "stepback", "vote x5")
):
    return [
        results[(p.id, s)]
        for p in problems
        for s in strategies
        if (p.id, s) in results
    ]

model: openai/gpt-oss-20b | ready: True
answers are remembered in: D:\Projects\GenAI-Agents-Work\.s15_cache.json


## 2. Token Overhead: The Hidden Cost of a One-Line Answer

Before comparing the four reasoning strategies on loan eligibility, start with a
simple numerical problem.

The warm-up uses a jacket priced at Rs 2,000:

- 20% discount
- another 10% discount on the reduced price
- Rs 100 coupon

The important point is that the second discount is applied **after** the first
discount. Therefore, the two discounts cannot simply be added together.

We will ask the model to solve this problem using:

1. **Plain** — return only the final answer.
2. **Chain-of-Thought (CoT)** — reason step by step before answering.

For each approach, the notebook records the answer, completion tokens,
reasoning tokens when reported by the provider, and elapsed time.

The purpose of this warm-up is to demonstrate that a short visible answer can
still involve additional model computation and token usage.

In [5]:
# Run the warm-up problem with the Plain and Chain-of-Thought strategies.

plain_demo = solve_plain(lab, DEMO)
cot_demo = solve_cot(lab, DEMO)

print("Warm-up problem:", DEMO.title)
print("Ground truth:", DEMO.truth)
print()

print("PLAIN")
print("Answer:", plain_demo.answer)
print("Completion tokens:", plain_demo.tokens)
print("Reasoning tokens:", plain_demo.reasoning_tokens)
print("Time (s):", round(plain_demo.seconds, 3))
print("Correct:", plain_demo.correct)

print()

print("CHAIN-OF-THOUGHT")
print("Answer:", cot_demo.answer)
print("Completion tokens:", cot_demo.tokens)
print("Reasoning tokens:", cot_demo.reasoning_tokens)
print("Time (s):", round(cot_demo.seconds, 3))
print("Correct:", cot_demo.correct)

Warm-up problem: Two discounts and a coupon
Ground truth: 1340.0

PLAIN
Answer: 1340.0
Completion tokens: 297
Reasoning tokens: 101
Time (s): 0.794
Correct: True

CHAIN-OF-THOUGHT
Answer: 1340.0
Completion tokens: 510
Reasoning tokens: 124
Time (s): 0.788
Correct: True


*Question 1 — Where did the output tokens go? Who paid for them, and could you see them? What does that mean for the phrase 'think step by step' on a model that already thinks?*

## 3. The Loan-Eligibility Checker: Plain vs. Chain-of-Thought (CoT)

The same loan policy is applied to ten applicant cases. Each case has an independently calculated ground-truth decision.

### Policy

A loan is approved only when all three conditions hold:

- Total monthly EMIs, including existing and new EMIs, are at most 40% of monthly income.
- Credit score is at least 700.
- Age at the end of the loan term is at most 60.

Otherwise, the application is declined.

### Strategies Tested

1. **Plain:** return only `FINAL: <answer>` with no working.
2. **Chain-of-Thought (CoT):** reason through the calculation and end with `FINAL: <answer>`.

### Prediction

Before running the experiment, I predicted that Plain might struggle with boundary conditions, age-at-term calculations, or the loan term expressed in months.

### Results

The actual results are taken directly from the experiment table below. The comparison focuses on whether CoT improves accuracy enough to justify its additional token and latency cost.

In [7]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")

predict_wrong = [
    "L04-boundaries",
    "L05-age-at-end",
    "L06-age-60",
    "L10-months",
]    # YOUR TURN: e.g. ["L05-age-at-end", "L10-months"]

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

In [8]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL))
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print("\nyou predicted plain would get wrong:", predict_wrong, "\nit actually got wrong:             ", got_wrong)
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       approve ok        approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       approve ok        approve ok        
L07-existing-emis   decline       decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       approve ok        approve ok        
L10-months          approve       approve ok        approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain        10/10         468       

*Question 2 — Put your predictions next to the table. Did step-by-step change the number of right answers? Did it change the cost? Which of the two moved more, and what does that say about when to pay for it?*

## 4. Step-Back Prompting: Principles First, Execution Second

**Step-back prompting** uses two calls:

1. **Principles:** identify the general rules, formulas, or traps without solving the specific case.
2. **Application:** solve the case using those principles.

This approach is useful when the main difficulty is identifying **which rules or framework should be applied**.

For this experiment, step-back is tested on the cases selected by the notebook based on the earlier results.

The key question is whether explicitly extracting the principles improves the final decision enough to justify the additional API call and token usage.

In [9]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print("\nWhat the model wrote when it was only allowed to step back (first case):\n")
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL)) # type: ignore
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L03-score-699', 'L04-boundaries', 'L08-two-fail']

What the model wrote when it was only allowed to step back (first case):

- **EMI‑to‑income check**: Sum all monthly EMIs (existing + new) and verify it does not exceed 40 % of the borrower’s monthly income.  
- **Credit‑score threshold**: The borrower’s credit score must be **≥ 700**.  
- **Age‑at‑term check**: Compute the borrower’s age at the *end* of the loan term; it must be **≤ 60 years**.  
- **All‑or‑nothing rule**: The loan is approved **only if every one of the three conditions is satisfied**; a single failure triggers a decline.  
- **Common pitfalls**:  
  - Forgetting to add existing EMIs to the new one.  
  - Using the borrower’s current age instead of the age at loan maturity.  
  - Mis‑applying the 40 % rule (e.g., rounding errors or using gross vs. net income).

case                right answer  plain             cot               stepback          
------------------------------------------------------

*Question 3 — Read the principles the model wrote. Were they useful, or did they just restate the policy you had already given it? When would you expect step-back to beat step-by-step, and when not?*

## 5. Self-Consistency: Buying Reliability with Majority Voting
Ask the same step-by-step question several times at a temperature above zero, so the answers can differ, and take the **most common answer**. It is how you buy reliability with money: five times the calls for one answer. We spend it on one case only, the one that gave the most trouble.

In [10]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}")
    single = results.get((hardest.id, "cot"))
    if single:
        print(f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L04-boundaries - Exactly on two limits

votes: {'approve': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,809 tokens, 5s   vs one step-by-step call: 792 tokens, 1s

spent so far: 32 calls (1 answered from memory), 21,685 tokens = about 11% of a 200,000-token free daily limit


*Question 4 — The tables give a cost per 1,000 questions for each strategy. Multiply it for 200,000 loan decisions a month. What would a 5x vote add to the monthly bill, and what size of accuracy gain would justify it? For which kinds of question would you never vote?*

## 6. The Dial: Controlling Reasoning Effort

Some reasoning models expose a `reasoning_effort` setting such as `low`, `medium`, or `high`.

The same problem is evaluated at different effort levels while keeping the task and prompt unchanged.

The purpose is to measure whether additional reasoning effort improves accuracy enough to justify increased token usage and latency.

In [11]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain") # type: ignore
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:
            print(f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}")
    if dial:
        print(comparison_table(dial, MODEL)) # type: ignore
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L04-boundaries        plain/low    approve  ok         1     316       53    0.6     $0.04
L04-boundaries        plain/high   approve  ok         1     591      328    0.8     $0.12

spent so far: 34 calls (1 answered from memory), 22,592 tokens = about 11% of a 200,000-token free daily limit


*Question 5 — If low and high gave the same answer, what did the extra thinking buy you? If they differed, which would you ship, and how would you know from one case that it was not luck?*

## 7. When Thinking Does Not Help: Logic vs. Arbitrary House Rules

Return to the complaint classifier from Session 14.

The experiment compares three approaches:

1. **Plain classification** — classify the complaint directly.
2. **Reasoning classification** — reason through the complaint before choosing a category.
3. **Few-shot classification** — use the four worked examples from Session 14 to provide the organization's house rules.

The bank's own convention (UPI problems are `account`, servicing requests are `other`). 
The key distinction is between **reasoning from known rules** and **learning an arbitrary organizational convention**.

If the model is never told that a particular servicing request belongs to `other`, additional reasoning cannot reliably discover that private rule. Explicit context or examples are required.

The purpose of this experiment is to determine whether additional reasoning improves classification when the difficulty comes from applying known rules, or whether the main problem is that the model simply needs the organization's examples and conventions.

In [12]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(lab, classify_spec(0), mode="plain"),
        "zero-shot + thinking": score_classifier_lab(lab, classify_cot_spec(), mode="cot"),
        "four examples, one word": score_classifier_lab(lab, classify_spec(4), mode="plain"),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}")
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,540    4.0
zero-shot + thinking         6/8     1,938    3.8
four examples, one word      6/8     3,197    3.6

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 58 calls (1 answered from memory), 29,267 

*Question 6 — Which change fixed the wrong labels: thinking, or examples? Why can no amount of thinking recover a rule the model was never told? What does that say about choosing between the two?*

## 8. Non-Reasoning Architecture: Local Model Benchmark

The same loan-eligibility cases are now evaluated using a **local chat model through Ollama**.

Unlike the reasoning model used in the earlier experiments, a standard local chat model does not provide the same hidden reasoning process. This makes it useful for testing whether explicitly asking the model to reason changes its performance.

The comparison focuses on:

- **Plain accuracy** — how often the model reaches the correct answer without visible reasoning.
- **Step-by-step accuracy** — whether explicitly requesting reasoning improves the result.
- **Token usage** — the additional output generated by the reasoning prompt.
- **Latency** — the time required to produce each answer.

The goal is to determine whether the benefit of Chain-of-Thought prompting depends on the underlying model architecture.

The results from this experiment should be interpreted using the measurements produced by the notebook rather than assumed in advance.

In [13]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter())
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

local model: llama3.2:3b
problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L02-ratio-narrow      plain        decline  ok         1     208        -   22.2         -
L02-ratio-narrow      cot          decline  ok         1     416        -   16.3         -
L05-age-at-end        plain        decline  ok         1     207        -    1.9         -
L05-age-at-end        cot          decline  ok         1     460        -   19.2         -
L07-existing-emis     plain        approve  WRONG      1     206        -    1.8         -
L07-existing-emis     cot          decline  ok         1     481        -   20.9         -
L09-large-numbers     plain        decline  WRONG      1     209        -    1.9         -
L09-large-numbers     cot          approve  ok         1     507        -   22.6         -
L10-months            plain        decline  WRONG      1     207 

*Question 7 — Compare the local model's plain and step-by-step rows with the hosted model's on the same cases. Which model gained more from step-by-step, and why? What does that tell you about advice such as 'always add think step by step'?*

## 9. Other Domains: Does Reasoning Help Beyond Loans?

The loan checker is only one type of multi-step problem. To see whether the value of reasoning generalizes, the notebook includes problems from several other domains.

These problems require the model to follow ordered rules, perform calculations, or avoid a specific trap.

The experiments cover:

- **Retail** — discounts, coupons, GST, and shipping rules
- **Insurance** — proportionate deduction, deductible, and co-pay
- **Probability** — applying conditional-probability calculations
- **Healthcare** — following an invented dosage/formulary rule
- **Billing** — calculating a prorated charge

Each problem has an independently calculated ground-truth answer, so the model's result can be evaluated objectively.

The purpose is not to decide that Chain-of-Thought is always better. Instead, compare the results and ask whether additional reasoning provides a useful accuracy improvement for different kinds of multi-step tasks.

The results below should be taken from the notebook's actual measurements rather than assumed in advance.

In [15]:
RUN_OTHER = True
if ready and RUN_OTHER:
    for p in OTHER + STRETCH:
        run(p, "plain")
        run(p, "cot")
    print(comparison_table(rows(OTHER + STRETCH, ("plain", "cot")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (set RUN_OTHER = True to run; needs a key)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
R1-retail-order       plain           5779  ok         1     584      290    1.3     $0.11
R1-retail-order       cot             5779  ok         1   1,028      322    1.3     $0.25
R2-insurance-claim    plain         126000  ok         1     675      403    1.0     $0.14
R2-insurance-claim    cot           126000  ok         1     901      282    1.1     $0.21
R4-defect-source      plain           37.5  ok         1     327      113    0.5     $0.05
R4-defect-source      cot             37.5  ok         1     694      139    0.7     $0.16
S1-dosage             plain            3.6  ok         1     389      171    0.7     $0.07
S1-dosage             cot              3.6  ok         1     655      166    1.0     $0.15
S2-telecom-proration  plain            254  ok         1     651      409    0.9     $0.14

## 10. Reasoning Behind a Curtain

In a real application, the customer should receive the **final answer**, not the model's private working.

This experiment separates the two parts of the response:

- `<scratchpad>` — private reasoning used internally by the model
- `<final>` — the customer-visible answer that is checked and stored

The notebook evaluates the `<final>` section only. The scratchpad is removed before the response is handled by the application.

This distinction matters because private reasoning can contain information that must never appear in the customer-facing response. Task 2 is a useful example because its internal credit score and cutoff are confidential.

We compare the original Session 14 prompt (**v1**) with a version that adds the scratchpad curtain (**v2**). Each version is run twice so that we can compare:

- acceptance-check results
- token usage
- whether confidential information appears in the scratchpad

The goal is not simply to make the model "think more", but to determine whether separating private reasoning from the final response provides a useful and safe application pattern.

In [16]:
BASE = {TASK1.id: TASK1.reference, TASK2.id: TASK2.reference}      # YOUR TURN (optional): put your own v1 specs here
print("What version 2 adds to Task 2's standing rules:\n")
print(with_scratchpad(BASE[TASK2.id]).to_messages(TASK2.user_input)[0]["content"])

What version 2 adds to Task 2's standing rules:

You are a customer-communications writer at a retail bank.

Rules:
- Use at most 280 characters.
- Address the customer by first name.
- Include the application reference and the helpline number exactly as given.
- Mention the reapply period.
- Never reveal internal scores, cutoffs or the internal reason.
- Do your reasoning inside <scratchpad>...</scratchpad> first. The scratchpad is private working and is never shown to anyone.

Output format: A <scratchpad> block, then <final>...</final> containing exactly this: One plain-text SMS. No subject line, no markdown, no emojis.


In [17]:
RUNS = 2
compare = {}
if ready:
    for task in (TASK1, TASK2):
        v1 = pass_rate(lab, task, BASE[task.id], runs=RUNS)
        v2 = pass_rate(lab, task, BASE[task.id], hidden_reasoning=True, runs=RUNS)
        compare[task.id] = (v1, v2)
        print(f"{task.id}")
        print(f"   v1: {v1['avg_checks']:.1f} of {v1['of']} checks, every check passed in {v1['all_passed']}/{RUNS} runs, {v1['avg_tokens']:,.0f} tokens a call")
        print(f"   v2: {v2['avg_checks']:.1f} of {v2['of']} checks, every check passed in {v2['all_passed']}/{RUNS} runs, {v2['avg_tokens']:,.0f} tokens a call")
        print("   ->", verdict(v1, v2))
        if v2["any_leak"]:
            print("   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer.")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

T1-credit-summary
   v1: 4.0 of 4 checks, every check passed in 2/2 runs, 1,077 tokens a call
   v2: 4.0 of 4 checks, every check passed in 2/2 runs, 1,316 tokens a call
   -> v1 and v2 pass the same number of checks (4.0 of 4): no reason to switch, at 1.2x the tokens per call.
T2-sms-declined
   v1: 6.0 of 6 checks, every check passed in 2/2 runs, 1,085 tokens a call
   v2: 6.0 of 6 checks, every check passed in 2/2 runs, 1,372 tokens a call
   -> v1 and v2 pass the same number of checks (6.0 of 6): no reason to switch, at 1.3x the tokens per call.

spent so far: 78 calls (1 answered from memory), 51,418 tokens = about 26% of a 200,000-token free daily limit


*Question 8 — Task 2's reply must never reveal the internal score. Did the scratchpad mention it? Who inside the bank may read a scratchpad, where would you store it, and for how long? Would you still keep the curtain if v2 gave no better check results?*

## 11. Work Repository

The experiments are useful only if the results can be reviewed later.

For this session, the work repository stores two kinds of evidence:

- `experiments/s15_accuracy_table.md` — the measured results for the ten loan cases.
- `prompts/library.md` — version 2 of the prompts that were improved during this session, while preserving the original version 1 entries.

The repository therefore contains both the **experimental measurements** and the **prompt history**.

Never overwrite an existing v1 prompt entry. Add v2 as a new entry so the evolution of the prompt remains visible in Git.

In [18]:
WORK_REPO = Path(r"D:\Projects\GenAI-Agents-Work")     # the same folder as Session 14
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

work repo: D:\Projects\GenAI-Agents-Work | exists: True


In [ ]:
SAVE_V2 = False
WHY = {
    TASK1.id: "v2 passed 4/4 checks in 2/2 runs like v1, but used 1,316 vs 1,077 tokens per call (1.2x).",
    TASK2.id: "v2 passed 6/6 checks in 2/2 runs like v1, but used 1,372 vs 1,085 tokens per call (1.3x).",
}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(WORK_REPO, task.id, 2, with_scratchpad(BASE[task.id]), run_v2["final"], run_v2["results"],
                        purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. " + WHY[task.id])
    table = accuracy_markdown(results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL)
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print("not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)")

saved 2 version-2 entries to D:\Projects\GenAI-Agents-Work\prompts\library.md
saved the accuracy table to D:\Projects\GenAI-Agents-Work\experiments\s15_accuracy_table.md


In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~ D:\Projects\GenAI-Agents-Work.
git add <file>
git status
git commit -m "Session 15: accuracy table and prompt library v2"
git push
```
Then run the self-check below. It reads your repository.

In [21]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

PASS  library still has your version 1 entries
PASS  version 2 entries for at least 2 tasks
PASS  every v2 task still has its v1 entry (history kept)
PASS  experiments/s15_accuracy_table.md exists
PASS  the table has all 10 loan cases


## 12. Decision Table
Fill this in from **your own numbers**, one word per cell: *use*, *skip*, or *only if* (and say what). It is the other deliverable of the session.

| Kind of question | plain | step by step | step back | vote x5 |
|---|---|---|---|---|
| Several rules chained, nothing hidden (L02, L07) | **use** | **skip** | **skip** | **only if high-stakes** |
| A rule that fails quietly (L05, L10) | **use** | **skip** | **only if uncertain** | **only if high-stakes** |
| Values exactly on a limit (L03, L04, L06) | **use** | **skip** | **skip** | **only if high-stakes** |
| A formula or a piece of theory to recall (R4) | **use** | **only if needed** | **only if rule unclear** | **skip** |
| Classification with your house rules (the complaints) | **skip** | **skip** | **skip** | **skip** |
| A calculation a calculator does exactly (S3, the EMI) | **skip** | **skip** | **skip** | **skip** |

## 13. Reflection
Answer questions 1-8 above, one or two lines each. Then one sentence: **when would you pay for thinking, and when would you refuse?**

1. The output tokens were paid for by the account using the model, and the visible completion included the model's response while some reasoning tokens were reported separately. Since the hosted model already reasons internally, asking it to "think step by step" added more visible output without improving the result.

2. My prediction was that Plain would get L04, L05, L06 and L10 wrong, but it actually got all 10 correct. Step-by-step also got 10/10, while increasing average tokens from 468 to 773 and cost from $0.08 to $0.18 per 1,000 questions. The extra cost increased more than the accuracy, so I would pay for it only when it provides a measurable accuracy benefit.

3. The step-back principles were useful because they highlighted the EMI, credit-score, age-at-term and all-or-nothing rules, but they mostly restated the policy already provided. I would expect step-back to help when the main difficulty is identifying which rules or formulas apply, but not when the rules are already clear.

4. On the tested case, voting did not improve the answer because all five samples were correct. The vote used 3,809 tokens and 5 calls compared with 792 tokens and one step-by-step call. At the observed cost, I would use voting only when a meaningful accuracy improvement is expected on a difficult or high-stakes decision, and I would avoid it for simple or deterministic calculations.

5. Low and high reasoning effort produced the same correct answer. High effort increased tokens from 316 to 591 and thinking tokens from 53 to 328, so it added more computation but no accuracy benefit on this case. I would test several representative cases before choosing high effort because one case alone cannot show that the result is not luck.

6. Thinking did not fix the wrong labels; the four examples did not fix them either. Both approaches remained at 6/8 because the model was never given the bank's rule that address and nominee updates belong to `other`. This shows that missing information should be supplied as context or examples rather than trying to solve the problem with more reasoning.

7. The local Llama 3.2 3B gained more from step-by-step prompting. It improved from 2/5 with Plain to 5/5 with CoT, while the hosted reasoning model stayed at 10/10 for both. This shows that prompting advice depends on the model, so "always add think step by step" is not a reliable rule.

8. The notebook's comparison did not report a scratchpad leak, so there was no flagged mention of the confidential values in the scratchpad. Access to a scratchpad should be restricted to the application components or authorised bank staff that genuinely need it, and it should be stored only for as long as required by the application's security and retention policy. I would still keep the curtain because the final response must not expose confidential information, even though v2 produced the same check results while using 1.2x–1.3x more tokens.

I would pay for thinking when testing shows that it gives a meaningful accuracy or reliability improvement on a difficult task; I would refuse the extra cost when the simpler prompt is already reliable or when an exact tool can perform the calculation.

Checkpoint: `uv run pytest tests/test_s15.py` is green, and the self-check above says `PASS` on all five lines. Push, and submit your repository link again.